# Weak-Label Evaluation

Measures `extract_weak_labels` (assertion-aware) against
`extract_weak_labels_naive` (the original, unmodified extractor) on the
58 human-labeled studies, and produces a per-label allowlist of which
labels are trustworthy enough to weak-label the remaining 4349
report-only studies with. Full design:
`docs/superpowers/specs/2026-08-09-weak-label-calibration-design.md`.

**This notebook is committed output-free, always** (not just until a
trusted run) -- its cells process real report text directly. Only
aggregate counts/metrics are ever printed; no report excerpts, no
per-study prediction tables, no study-identifier lists.

In [ ]:
import random
import sys
from pathlib import Path

import numpy as np
import pandas as pd

NOTEBOOK_VERSION = "v1"
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

IS_KAGGLE = Path("/kaggle/input").exists()

if IS_KAGGLE:
    DATA_DIR = Path("/kaggle/input/competitions/rsna-knee-abnormality-detection")

    SRC_DATASET_DIR = Path("/kaggle/input/datasets/tuannm3812/rsna-knee-mri-src")
    _src_candidates = (SRC_DATASET_DIR / "src", SRC_DATASET_DIR)
    _src_root = next((c for c in _src_candidates if (c / "knee_mri").is_dir()), None)
    if _src_root is None:
        raise RuntimeError(
            f"knee_mri package not found under any of {[str(c) for c in _src_candidates]} "
            "-- check the rsna-knee-mri-src dataset's actual mounted layout "
            "(see docs/0_coding_standards.md, 'Pushing Notebooks To Kaggle') "
            "and record it in docs/6_kaggle_troubleshooting.md."
        )
    sys.path.insert(0, str(_src_root))
else:
    raise RuntimeError(
        "This notebook only runs on Kaggle -- see docs/0_coding_standards.md "
        "('Data & Compute'): the competition dataset is never downloaded "
        "locally."
    )

print(f"NOTEBOOK_VERSION={NOTEBOOK_VERSION}")
print(f"IS_KAGGLE={IS_KAGGLE}")
print(f"DATA_DIR={DATA_DIR}")
print(f"SRC_ROOT={_src_root}")

## 1. Load the 58 labeled studies

In [ ]:
from knee_mri.dataset import split_labeled_studies

train_df = pd.read_csv(DATA_DIR / "train.csv")
labeled_df, unlabeled_df = split_labeled_studies(train_df)

print(f"Labeled studies: {len(labeled_df)}")
print(f"Unlabeled (report-only) studies: {len(unlabeled_df)}")

## 2. Baseline measurement (naive, pre-fix extractor)

In [ ]:
from knee_mri.labels import extract_weak_labels, extract_weak_labels_naive
from knee_mri.weak_label_evaluation import weak_label_metrics

baseline_metrics = weak_label_metrics(labeled_df, extract_weak_labels_naive)
print(baseline_metrics)

**Insight:** pending first Kaggle run.

## 3. Fixed measurement (assertion-aware extractor)

In [ ]:
fixed_metrics = weak_label_metrics(labeled_df, extract_weak_labels)
print(fixed_metrics)

**Insight:** pending first Kaggle run.

## 4. Error taxonomy (resolver diagnostics, counts only)

In [ ]:
from collections import Counter

from knee_mri.labels import LABEL_COLUMNS, _resolution_signature, _resolve_weak_labels
from knee_mri.weak_label_evaluation import orthographic_bucket

taxonomy_counts = Counter()
for _, row in labeled_df.iterrows():
    bucket = orthographic_bucket(row["Report"])
    resolutions = _resolve_weak_labels(row["Report"])
    for label in LABEL_COLUMNS:
        resolution = resolutions[label]
        truth = row[label]
        prediction = resolution.value
        is_error = (prediction == 1 and truth == 0) or (prediction != 1 and truth == 1)
        if not is_error:
            continue
        prediction_error = "false_positive" if prediction == 1 else "false_negative"
        signature = _resolution_signature(resolution.mentions)
        taxonomy_counts[(label, bucket, signature, prediction_error)] += 1

# Counts only -- never report text, matched text, or per-study identifiers.
for key, count in sorted(taxonomy_counts.items()):
    print(key, count)

**Insight:** pending first Kaggle run. Any causal hypothesis about these
counts belongs in `docs/4_experiments.md`, explicitly hedged as
unconfirmed -- `resolution_signature` and `prediction_error` are
directly observed facts, not an explanation of why the human label
disagrees.

## 5. Orthographic-bucket comparison (labeled vs. all unlabeled studies)

In [ ]:
labeled_buckets = (
    labeled_df["Report"].dropna().apply(orthographic_bucket).value_counts(normalize=True)
)
unlabeled_buckets = (
    unlabeled_df["Report"].dropna().apply(orthographic_bucket).value_counts(normalize=True)
)

comparison = pd.DataFrame({"labeled": labeled_buckets, "unlabeled": unlabeled_buckets}).fillna(0.0)
print(comparison)

**Insight:** pending first Kaggle run. If the labeled set's bucket mix
doesn't resemble the unlabeled set's, that's a caveat on how far the
allowlist below generalizes -- not assumed to transfer automatically.

## 6. Per-label allowlist

In [ ]:
allowlist = fixed_metrics[fixed_metrics["passes_gate"]].index.tolist()
print(f"Allowlist ({len(allowlist)}/{len(LABEL_COLUMNS)}): {allowlist}")

**Insight:** pending first Kaggle run. This allowlist -- not a single
GO/NO-GO flag -- is this notebook's deliverable: any future work that
applies weak labels to expand a training set must only use labels on
this list.